# Writing Project Tools - Google Colab Template

Run this notebook in Google Colab to clone the toolkit from GitHub, install it in the runtime, scaffold a writing-project workspace, and generate Word documents from Markdown.

Default GitHub repo: `https://github.com/ruchirlives/src.git`

Use `Runtime > Run all` for a fresh setup. Colab runtimes are temporary unless you mount Google Drive and place the project workspace there.

## 1. Settings

Edit these values before running if you want a different branch, repository URL, or project folder name.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ruchirlives/src.git"
REPO_BRANCH = "main"
TOOLKIT_DIR = Path("/content/writing-project-tools")
PROJECT_DIR = Path("/content/writing-project")

print(f"Toolkit repo: {REPO_URL}")
print(f"Toolkit path: {TOOLKIT_DIR}")
print(f"Writing project path: {PROJECT_DIR}")

## 2. Optional: Use Google Drive For Persistent Work

Run this cell if you want the scaffolded article folder to persist after the Colab runtime shuts down. Skip it if `/content/writing-project` is fine for temporary work.

In [ ]:
USE_GOOGLE_DRIVE = False
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/writing-project"

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
    print(f"Using persistent project path: {PROJECT_DIR}")
else:
    print(f"Using temporary project path: {PROJECT_DIR}")

## 3. Clone Or Update The Toolkit Repo

This fetches the GitHub repository into the Colab runtime. Re-run after pushing changes to GitHub.

In [ ]:
import subprocess

def run(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

if TOOLKIT_DIR.exists():
    run(["git", "fetch", "origin"], cwd=TOOLKIT_DIR)
    run(["git", "checkout", REPO_BRANCH], cwd=TOOLKIT_DIR)
    run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=TOOLKIT_DIR)
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(TOOLKIT_DIR)])

## 4. Install The Toolkit

Editable install means notebook changes can use the cloned source immediately.

In [ ]:
import sys

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-e", str(TOOLKIT_DIR)])

## 5. Scaffold A Writing Project Workspace

This creates the Markdown source files, including `article-plan.md`, `article-outline.md`, `article-draft.md`, `assertions.csv`, `docx_sources.csv`, and project README. Existing files are left alone unless `FORCE_SCAFFOLD` is set to `True`.

In [ ]:
FORCE_SCAFFOLD = False

command = [sys.executable, str(TOOLKIT_DIR / "scaffold_article.py"), str(PROJECT_DIR)]
if FORCE_SCAFFOLD:
    command.append("--force")

run(command)
print("\nProject files:")
for path in sorted(PROJECT_DIR.iterdir()):
    print("-", path.name)

## 6. Edit Project Markdown In Colab

Use the file browser on the left, or edit from cells like this. The scaffold creates `article-plan.md`, `article-outline.md`, `article-draft.md`, source notes, context, considerations, colleagues, and author instructions.

In [ ]:
plan_path = PROJECT_DIR / "article-plan.md"
print(plan_path.read_text(encoding="utf-8"))

## 7. Generate Word Documents

The output paths are controlled by `docx_sources.csv`. By default, generated `.docx` files for the plan, outline, and draft go into `docs/` inside the project folder.

In [ ]:
run(["create-writing-docs"], cwd=PROJECT_DIR)

docs_dir = PROJECT_DIR / "docs"
print("\nGenerated documents:")
for path in sorted(docs_dir.glob("*.docx")):
    print(path)

## 8. Download Generated Files

Use this to download generated `.docx` files from the Colab runtime.

In [ ]:
from google.colab import files

for path in sorted((PROJECT_DIR / "docs").glob("*.docx")):
    files.download(str(path))

## 9. Project Editor In Colab

The local `edit-assertions` web server includes both assertions review and Markdown editing. In Colab, run it with a public port proxy URL using `google.colab.output.eval_js`; open `/markdown` on that proxy URL to edit project `.md` files.

In [ ]:
import os
import signal
import subprocess
import time
from google.colab import output

PORT = 8765
ASSERTIONS_PROCESS = None

if ASSERTIONS_PROCESS is not None and ASSERTIONS_PROCESS.poll() is None:
    os.kill(ASSERTIONS_PROCESS.pid, signal.SIGTERM)

ASSERTIONS_PROCESS = subprocess.Popen(
    ["edit-assertions", "--host", "0.0.0.0", "--port", str(PORT), "--no-open"],
    cwd=PROJECT_DIR,
)
time.sleep(1)

print("Project editor URL:")
print(output.eval_js(f"google.colab.kernel.proxyPort({PORT})"))
print("Open /markdown on that URL for the Markdown editor.")
print("Stop the server with the next cell when finished.")

In [ ]:
if 'ASSERTIONS_PROCESS' in globals() and ASSERTIONS_PROCESS is not None and ASSERTIONS_PROCESS.poll() is None:
    ASSERTIONS_PROCESS.terminate()
    print("Assertions editor stopped.")
else:
    print("Assertions editor is not running.")